In [2]:
import torch
import torch.nn as nn
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

In [3]:
# ---------------------------------------------------------------------
# 1. Génération des données d'observation (Vérité terrain)
# ---------------------------------------------------------------------
# Les paramètres réels sont définis : rho=10, sigma=15, beta=8/3.
def lorenz_true(t, state, rho=10.0, sigma=15.0, beta=8.0/3.0):
    x, y, z = state
    return [rho * (y - x), 
            x * (sigma - z) - y, 
            x * y - beta * z]

# Condition initiale et domaine temporel[cite: 450, 451].
t_span = (0.0, 3.0)
y0 = [-8.0, 7.0, 27.0]

# Création de 25 points d'observation équispacés.
t_eval = np.linspace(t_span[0], t_span[1], 25)
sol = solve_ivp(lorenz_true, t_span, y0, t_eval=t_eval)

# Conversion en tenseurs PyTorch
t_obs = torch.tensor(sol.t, dtype=torch.float32).view(-1, 1)
y_obs = torch.tensor(sol.y.T, dtype=torch.float32)

# Points résiduels (physique) : 400 points uniformément distribués.
# requires_grad=True est essentiel pour calculer les dérivées par rapport à t.
t_physics = torch.empty(400, 1).uniform_(t_span[0], t_span[1]).requires_grad_(True)

In [4]:
# ---------------------------------------------------------------------
# 2. Définition du réseau PINN
# ---------------------------------------------------------------------
class LorenzPINN(nn.Module):
    def __init__(self):
        super().__init__()
        # Architecture : 1 entrée (t), 3 couches cachées de 40 neurones, 3 sorties (x, y, z).
        # Fonction d'activation : tanh.
        self.net = nn.Sequential(
            nn.Linear(1, 40), nn.Tanh(),
            nn.Linear(40, 40), nn.Tanh(),
            nn.Linear(40, 40), nn.Tanh(),
            nn.Linear(40, 3)
        )
        # Définition des paramètres inverses comme variables entraînables
        self.rho = nn.Parameter(torch.tensor([1.0]))
        self.sigma = nn.Parameter(torch.tensor([1.0]))
        self.beta = nn.Parameter(torch.tensor([1.0]))

    def forward(self, t):
        return self.net(t)

# Instanciation du modèle
model = LorenzPINN()

In [5]:
# ---------------------------------------------------------------------
# 3. Configuration de l'entraînement
# ---------------------------------------------------------------------
# Optimiseur Adam avec un taux d'apprentissage de 0.001.
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Nombre d'itérations.
epochs = 60000

print("Début de l'entraînement...")

for epoch in range(epochs):
    optimizer.zero_grad()
    
    # --- A. Perte sur les données d'observation (Data Loss) ---
    pred_obs = model(t_obs)
    loss_data = torch.mean((pred_obs - y_obs) ** 2)
    
    # --- B. Perte sur la physique (Physics/Residual Loss) ---
    pred_phys = model(t_physics)
    x = pred_phys[:, 0:1]
    y = pred_phys[:, 1:2]
    z = pred_phys[:, 2:3]
    
    # Calcul des dérivées temporelles avec torch.autograd.grad
    dx_dt = torch.autograd.grad(x, t_physics, grad_outputs=torch.ones_like(x), create_graph=True)[0]
    dy_dt = torch.autograd.grad(y, t_physics, grad_outputs=torch.ones_like(y), create_graph=True)[0]
    dz_dt = torch.autograd.grad(z, t_physics, grad_outputs=torch.ones_like(z), create_graph=True)[0]
    
    # Résidus des équations de Lorenz
    eq_x = dx_dt - model.rho * (y - x)
    eq_y = dy_dt - x * (model.sigma - z) + y
    eq_z = dz_dt - x * y + model.beta * z
    
    # Erreur quadratique moyenne des résidus
    loss_phys = torch.mean(eq_x**2 + eq_y**2 + eq_z**2)
    
    # --- C. Perte totale et rétropropagation ---
    # Les poids wf, wb et wi sont fixés à 1.
    loss = loss_data + loss_phys
    loss.backward()
    optimizer.step()
    
    # Affichage de l'évolution
    if epoch % 5000 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch:5d} | Loss: {loss.item():.4e} | "
              f"rho: {model.rho.item():.3f} | sigma: {model.sigma.item():.3f} | beta: {model.beta.item():.3f}")

print("\n--- Résultats Finaux ---")
print("Valeurs réelles      : rho = 10.000, sigma = 15.000, beta = 2.667")
print(f"Valeurs identifiées  : rho = {model.rho.item():.3f}, sigma = {model.sigma.item():.3f}, beta = {model.beta.item():.3f}")

Début de l'entraînement...
Epoch     0 | Loss: 9.8652e+01 | rho: 0.999 | sigma: 1.001 | beta: 0.999
Epoch  5000 | Loss: 1.5885e+01 | rho: 0.038 | sigma: -0.040 | beta: -0.036
Epoch 10000 | Loss: 1.5675e+01 | rho: 0.033 | sigma: -3.869 | beta: -0.014
Epoch 15000 | Loss: 1.5519e+01 | rho: 0.027 | sigma: -8.143 | beta: 0.003
Epoch 20000 | Loss: 1.5410e+01 | rho: 0.023 | sigma: -12.272 | beta: 0.016
Epoch 25000 | Loss: 1.5327e+01 | rho: 0.019 | sigma: -16.221 | beta: 0.026
Epoch 30000 | Loss: 1.5261e+01 | rho: 0.017 | sigma: -19.969 | beta: 0.034


KeyboardInterrupt: 

In [ ]:
# ---------------------------------------------------------------------
# 4. Visualisation des résultats
# ---------------------------------------------------------------------

# --- 1. Génération d'une grille temporelle fine pour un affichage lisse ---
t_plot = np.linspace(t_span[0], t_span[1], 500)
sol_plot = solve_ivp(lorenz_true, t_span, y0, t_eval=t_plot)

# --- 2. Prédictions du modèle sur cette grille ---
# Passage du modèle en mode évaluation
model.eval() 
t_tensor = torch.tensor(t_plot, dtype=torch.float32).view(-1, 1)

with torch.no_grad():
    pred_plot = model(t_tensor).numpy()

# --- 3. Création des graphiques ---
fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
labels = ['x', 'y', 'z']
colors = ['tab:red', 'tab:green', 'tab:blue']

for i in range(3):
    # Tracé de la solution exacte (ligne continue noire)
    axs[i].plot(t_plot, sol_plot.y[i], label=f'{labels[i]} exacte', color='black', linewidth=2)
    
    # Tracé de la prédiction du PINN (ligne pointillée colorée)
    axs[i].plot(t_plot, pred_plot[:, i], '--', label=f'{labels[i]} PINN', color=colors[i], linewidth=2.5)
    
    # Ajout des 25 points d'entraînement (marqueurs)
    axs[i].scatter(sol.t, sol.y[i], color='orange', edgecolor='black', s=50, zorder=5, label='Observations')
    
    axs[i].set_ylabel(f'Composante {labels[i]}', fontsize=12)
    axs[i].legend(loc='upper right')
    axs[i].grid(True, linestyle=':', alpha=0.7)

axs[-1].set_xlabel('Temps (t)', fontsize=12)
plt.suptitle(f"Problème Inverse de Lorenz : Prédiction vs Réalité\n"
             f"Identifiés : $\\rho$={model.rho.item():.2f}, $\\sigma$={model.sigma.item():.2f}, $\\beta$={model.beta.item():.2f}", 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()